# 🧠 Stage 1: Benchmark Ingestion & Verification (Clean Architecture)
**Project:** Reduction Ladder for Code & Multi-Arm Mitigation  
**Organization:** Orange Innovation Labs  
**Authors:** Omar Abdelhamid, Nour Walid  
**Supervisor:** Dr. Ghada  

---

### 🎯 Objectives:
1. Ingest **L0 to L5** benchmark datasets using the decoupled `HuggingFaceBenchmarkLoader`.
2. Standardize all problem schemas into domain `BenchmarkTask` entities.
3. Execute sandbox verification on canonical ground-truth solutions via `DataService`.
4. Inspect samples across all 6 ladder levels.

In [ ]:
import os
import sys
import pandas as pd

# Ensure project root is on sys.path
sys.path.insert(0, os.path.abspath(".."))

from src.services.data_service import DataService
from src.infrastructure.hf_loader import HuggingFaceBenchmarkLoader, LADDER_DATASET_CONFIGS
from src.infrastructure.sandbox import MultiprocessSandbox
from src.core.entities import BenchmarkTask

data_service = DataService(loader=HuggingFaceBenchmarkLoader(cache_dir="../data/ladder"))
print("✅ Clean Architecture DataService initialized successfully!")

## 1. Overview of the Reduction Ladder Levels
Let's inspect the target benchmarks and their Hugging Face sources.

In [ ]:
ladder_df = pd.DataFrame.from_dict(LADDER_DATASET_CONFIGS, orient="index")
ladder_df.index.name = "Level"
ladder_df

## 2. Ingest and Normalize All Ladder Levels (L0 to L5)

In [ ]:
all_ladder_data = data_service.prepare_all_benchmarks(force_download=False)

print(f"\n🎉 All {len(all_ladder_data)} levels loaded as domain entities!")

## 3. Ground-Truth Sandbox Verification Suite
Run unit tests on all canonical solutions to ensure 100% test-suite correctness.

In [ ]:
verification_results = {}
for level_key, tasks in all_ladder_data.items():
    print(f"\n==================== Verifying {level_key} ====================")
    res = data_service.verify_ground_truth(tasks)
    verification_results[level_key] = res

summary_df = pd.DataFrame([
    {"Level": k, "Total Tasks": v["total"], "Passed": v["passed"], "Pass Rate (%)": v["pass_rate"]}
    for k, v in verification_results.items()
])
summary_df

## 4. Visual Inspection: Comparing Problem Transformations Across Levels

In [ ]:
for level in ["L0", "L1", "L2", "L3", "L4", "L5"]:
    task = all_ladder_data[level][0]
    print(f"\n{'='*30} [{level}]: {task.benchmark} {'='*30}")
    print(f"Task ID: {task.task_id}")
    print("Prompt:")
    print(task.prompt[:300] + "...\n")